In [1]:
import os
import gc
import numpy as np
import pandas as pd
import random
from sklearn.ensemble import HistGradientBoostingRegressor


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


dtypes = {
    "R": "int8",
    "C": "int8",
    "breath_id": "int32",
    "id": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}




In [2]:
train_path = "../input/ventilator-pressure-prediction/train.csv"
test_path = "../input/ventilator-pressure-prediction/test.csv"
sample_sub_path = "../input/ventilator-pressure-prediction/sample_submission.csv"

feature_cols = [
    "R",
    "C",
    "time_step",
    "u_in",
    "u_out",
    "breath_id",
    "id",
]
train_usecols = feature_cols + ["pressure"]
test_usecols = feature_cols

train_df = pd.read_csv(train_path, dtype=dtypes, usecols=train_usecols)
test_df = pd.read_csv(test_path, dtype=dtypes, usecols=test_usecols)

# basic interaction / non‑linear features
train_df["RC"] = train_df["R"] * train_df["C"]
test_df["RC"] = test_df["R"] * test_df["C"]
train_df["u_in_sq"] = train_df["u_in"] ** 2
test_df["u_in_sq"] = test_df["u_in"] ** 2

# first‑order lag features (already present in original code)
g_train = train_df.groupby("breath_id", sort=False)
train_df["u_in_lag"] = g_train["u_in"].shift().fillna(0)
train_df["u_out_lag"] = g_train["u_out"].shift().fillna(0)
train_df["time_step_diff"] = g_train["time_step"].diff().fillna(0)
train_df["cum_u_in"] = g_train["u_in"].cumsum()

g_test = test_df.groupby("breath_id", sort=False)
test_df["u_in_lag"] = g_test["u_in"].shift().fillna(0)
test_df["u_out_lag"] = g_test["u_out"].shift().fillna(0)
test_df["time_step_diff"] = g_test["time_step"].diff().fillna(0)
test_df["cum_u_in"] = g_test["u_in"].cumsum()

# **new cheap second‑order lag features**
train_df["u_in_lag2"] = g_train["u_in"].shift(2).fillna(0)
train_df["u_out_lag2"] = g_train["u_out"].shift(2).fillna(0)
test_df["u_in_lag2"] = g_test["u_in"].shift(2).fillna(0)
test_df["u_out_lag2"] = g_test["u_out"].shift(2).fillna(0)

feature_cols = [
    "R",
    "C",
    "time_step",
    "u_in",
    "u_out",
    "breath_id",
    "id",
    "RC",
    "u_in_sq",
    "u_in_lag",
    "u_out_lag",
    "time_step_diff",
    "cum_u_in",
    "u_in_lag2",  # new
    "u_out_lag2",  # new
]

X_train = np.ascontiguousarray(train_df[feature_cols].to_numpy(dtype=np.float32))
y_train = train_df["pressure"].to_numpy(dtype=np.float32)

X_test = np.ascontiguousarray(test_df[feature_cols].to_numpy(dtype=np.float32))

p_min, p_max = y_train.min(), y_train.max()

set_seed(2021)

del train_df, test_df, g_train, g_test
gc.collect()




23

In [3]:
model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.03,  # smaller LR for finer learning
    max_iter=2000,  # allow more trees
    loss="absolute_error",
    random_state=2021,
    early_stopping=True,
    n_iter_no_change=10,
    validation_fraction=0.1,
    max_bins=255,  # default bins → better resolution
    verbose=0,
)

model.fit(X_train, y_train)

raw_preds = model.predict(X_test)

del X_train, y_train, X_test
gc.collect()




0

In [4]:
raw_preds = np.clip(raw_preds, p_min, p_max)

submission = pd.read_csv(sample_sub_path)
submission["pressure"] = raw_preds.astype(np.float32)

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
